In [11]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from shapely.ops import nearest_points
from shapely.strtree import STRtree
import time
import logging
import sys
from datetime import datetime

# ── 0. Logging setup ───────────────────────────────────────────────────────────
log_path = r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\processing_log.txt"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(log_path, mode="w", encoding="utf-8"),
        logging.StreamHandler(sys.stdout)
    ]
)
log = logging.getLogger()

def log_elapsed(stage_start, label):
    elapsed = time.time() - stage_start
    log.info(f"  [DONE] {label} completed in {elapsed:.1f}s")
    return elapsed

script_start = time.time()
log.info("=" * 60)
log.info("  CONUS ROW -> Transmission Line Distance Script")
log.info(f"  Run started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log.info("=" * 60)

# ── 1. Load data ───────────────────────────────────────────────────────────────
log.info("\n[STEP 1] Loading shapefiles...")
t = time.time()

row_gdf = gpd.read_file(r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\Alaska\Alaska_ROW_Centroids.shp")
log.info(f"  ROW points loaded:            {len(row_gdf):,} records")

tl_gdf = gpd.read_file(r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\TransLines_InService_NotAvailable.shp")
log.info(f"  Transmission lines loaded:    {len(tl_gdf):,} records")
log_elapsed(t, "Data load")

# ── 2. Null geometry check ─────────────────────────────────────────────────────
log.info("\n[STEP 2] Checking for null geometries...")
t = time.time()

row_nulls = row_gdf.geometry.isna().sum()
tl_nulls  = tl_gdf.geometry.isna().sum()
log.info(f"  Null geometries in ROW:                {row_nulls:,}")
log.info(f"  Null geometries in Transmission Lines: {tl_nulls:,}")

row_gdf = row_gdf[row_gdf.geometry.notna()].reset_index(drop=True)
tl_gdf  = tl_gdf[tl_gdf.geometry.notna()].reset_index(drop=True)
log.info(f"  ROW points after cleaning:             {len(row_gdf):,} records")
log.info(f"  Transmission lines after cleaning:     {len(tl_gdf):,} records")
log_elapsed(t, "Null check + clean")

# ── 3. Reproject ───────────────────────────────────────────────────────────────
# EPSG:5070 = NAD83 Conus Albers - equal-area meters, ideal for CONUS distance work
log.info("\n[STEP 3] Reprojecting to EPSG:  ...")
t = time.time()

CRS = "EPSG:3338"
row_gdf = row_gdf.to_crs(CRS)
tl_gdf  = tl_gdf.to_crs(CRS)
log.info(f"  ROW CRS:   {row_gdf.crs}")
log.info(f"  Lines CRS: {tl_gdf.crs}")
log_elapsed(t, "Reprojection")

# ── 4. Build STRtree spatial index ─────────────────────────────────────────────
log.info("\n[STEP 4] Building STRtree spatial index on transmission lines...")
t = time.time()

tl_geoms = tl_gdf.geometry.values
tl_tree  = STRtree(tl_geoms)
log.info(f"  Spatial index built from {len(tl_geoms):,} line geometries")
log_elapsed(t, "STRtree build")

# ── 5. Snap ROW points to nearest transmission line ────────────────────────────
log.info(f"\n[STEP 5] Snapping {len(row_gdf):,} ROW points to nearest transmission line...")
log.info(f"  Strategy: STRtree.nearest -> nearest_points exact snap")
log.info(f"  Progress logged every 1,000 records")
t       = time.time()
t_batch = time.time()

nearest_pts = []
distances   = []
BATCH       = 1000

for i, row_pt in enumerate(row_gdf.geometry):

    candidate_idx  = tl_tree.nearest(row_pt)
    candidate_line = tl_geoms[candidate_idx]
    snap_pt        = nearest_points(row_pt, candidate_line)[1]
    dist           = row_pt.distance(snap_pt)

    nearest_pts.append(snap_pt)
    distances.append(dist)

    if (i + 1) % BATCH == 0:
        batch_elapsed = time.time() - t_batch
        total_elapsed = time.time() - t
        pct_done      = (i + 1) / len(row_gdf)
        est_remaining = (total_elapsed / pct_done) * (1 - pct_done)

        log.info(
            f"  [{i+1:>6,} / {len(row_gdf):,}]  "
            f"{pct_done*100:5.1f}% complete  |  "
            f"batch: {batch_elapsed:.1f}s  |  "
            f"elapsed: {total_elapsed:.1f}s  |  "
            f"est. remaining: {est_remaining:.1f}s"
        )
        t_batch = time.time()

distances    = np.array(distances)
snap_elapsed = time.time() - t
log.info(f"  Snap complete")
log.info(f"  Total snap time:  {snap_elapsed:.1f}s  ({snap_elapsed/60:.2f} min)")
log.info(f"  Avg time/point:   {snap_elapsed/len(row_gdf)*1000:.2f} ms")
log.info(f"  Min distance:     {distances.min():,.2f} m")
log.info(f"  Max distance:     {distances.max():,.2f} m")
log.info(f"  Mean distance:    {distances.mean():,.2f} m")
log.info(f"  Median distance:  {np.median(distances):,.2f} m")

# ── 6. Build connecting lines ──────────────────────────────────────────────────
log.info("\n[STEP 6] Building connecting lines...")
t = time.time()

lines = [
    LineString([row_gdf.geometry.iloc[i], nearest_pts[i]])
    for i in range(len(row_gdf))
]
log.info(f"  {len(lines):,} lines created")
log_elapsed(t, "Line construction")

# ── 7. Export CSV ──────────────────────────────────────────────────────────────
log.info("\n[STEP 7] Exporting CSV...")
t = time.time()

output_df = pd.DataFrame({
    "Facility_I":               row_gdf["Facility_I"],
    "TransLine_Distance_Meters": np.round(distances, 2)
})
csv_path = r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\Alaska\Alaska_ROW_TransLine_distances.csv"
output_df.to_csv(csv_path, index=False)
log.info(f"  Rows exported: {len(output_df):,}")
log.info(f"  Path: {csv_path}")
log_elapsed(t, "CSV export")

# ── 8. Export lines shapefile ──────────────────────────────────────────────────
log.info("\n[STEP 8] Exporting lines shapefile...")
t = time.time()

lines_gdf = gpd.GeoDataFrame(output_df, geometry=lines, crs=CRS)
shp_path = r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\Alaska\Alaska_ROW_to_TransLine.shp"
lines_gdf.to_file(shp_path)
log.info(f"  Features exported: {len(lines_gdf):,}")
log.info(f"  Path: {shp_path}")
log_elapsed(t, "Shapefile export")

# ── 9. Final summary ───────────────────────────────────────────────────────────
total_time = time.time() - script_start
log.info("\n" + "=" * 60)
log.info("  FINAL SUMMARY")
log.info("=" * 60)
log.info(f"  Run finished:                {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log.info(f"  Total script time:           {total_time:.1f}s  ({total_time/60:.2f} min)")
log.info(f"  Total ROW points processed:  {len(output_df):,}")
log.info(f"  Null ROW geometries dropped: {row_nulls:,}")
log.info(f"  Null TL geometries dropped:  {tl_nulls:,}")
log.info(f"  Min distance:                {distances.min():,.2f} m")
log.info(f"  Max distance:                {distances.max():,.2f} m")
log.info(f"  Mean distance:               {distances.mean():,.2f} m")
log.info(f"  Median distance:             {np.median(distances):,.2f} m")
log.info(f"  CSV output:                  {csv_path}")
log.info(f"  Shapefile output:            {shp_path}")
log.info(f"  Log file:                    {log_path}")
log.info("=" * 60)

print(output_df.head(10))

2026-04-15 16:27:41  ============================================================
2026-04-15 16:27:41    CONUS ROW -> Transmission Line Distance Script
2026-04-15 16:27:41    Run started: 2026-04-15 16:27:41
2026-04-15 16:27:41  ============================================================
2026-04-15 16:27:41  
[STEP 1] Loading shapefiles...
2026-04-15 16:27:41    ROW points loaded:            11 records
2026-04-15 16:27:44    Transmission lines loaded:    94,098 records
2026-04-15 16:27:44    [DONE] Data load completed in 3.5s
2026-04-15 16:27:44  
[STEP 2] Checking for null geometries...
2026-04-15 16:27:44    Null geometries in ROW:                0
2026-04-15 16:27:44    Null geometries in Transmission Lines: 0
2026-04-15 16:27:44    ROW points after cleaning:             11 records
2026-04-15 16:27:44    Transmission lines after cleaning:     94,098 records
2026-04-15 16:27:44    [DONE] Null check + clean completed in 0.1s
2026-04-15 16:27:44  
[STEP 3] Reprojecting to EPSG:  ...
2

C:\Users\KyleSteen.AzureAD\AppData\Local\Temp\ipykernel_24472\1339528947.py:161: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  lines_gdf.to_file(shp_path)
C:\Users\KyleSteen.AzureAD\miniconda3\envs\myenv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'TransLine_Distance_Meters' to 'TransLine_'
  ogr_write(


In [12]:
import geopandas as gpd
import pandas as pd

# ── INPUT PATHS ────────────────────────────────────────────────────────────────
fc_path = r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\Alaska\Alaska_Level_2_Analysis_AEP_GlintGlare_LCOE_Shape_Size_Substations.gpkg"
csv_path = r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\Alaska\Alaska_ROW_TransLine_distances.csv"

output_path = r"C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\Alaska\Alaska_output.gpkg"

# ── 1. LOAD DATA ───────────────────────────────────────────────────────────────
print("Loading data...")

fc_gdf = gpd.read_file(fc_path)
csv_df = pd.read_csv(csv_path)

print(f"  Feature class rows: {len(fc_gdf):,}")
print(f"  CSV rows:           {len(csv_df):,}")

# ── 2. CLEAN KEYS ──────────────────────────────────────────────────────────────
print("\nCleaning keys...")

fc_gdf["Facility_ID"] = fc_gdf["Facility_ID"].astype(str).str.strip()
csv_df["Facility_ID"] = csv_df["Facility_ID"].astype(str).str.strip()

# remove trailing .0 issues
fc_gdf["Facility_ID"] = fc_gdf["Facility_ID"].str.replace(r"\.0$", "", regex=True)
csv_df["Facility_ID"] = csv_df["Facility_ID"].str.replace(r"\.0$", "", regex=True)

# ensure one value per Facility_ID
csv_df = csv_df.drop_duplicates(subset="Facility_ID")

print(f"  Unique CSV keys: {len(csv_df):,}")

# ── 3. BUILD LOOKUP ────────────────────────────────────────────────────────────
print("\nBuilding lookup table...")

lookup = dict(zip(
    csv_df["Facility_ID"],
    csv_df["TransLine_Distance_Meters"]   # ✅ UPDATED FIELD
))

# ── 4. APPLY JOIN ──────────────────────────────────────────────────────────────
print("Applying distance values...")

fc_gdf["TransLine_Distance_Meters"] = fc_gdf["Facility_ID"].map(lookup)

# ── 5. CHECK RESULTS ───────────────────────────────────────────────────────────
missing = fc_gdf["TransLine_Distance_Meters"].isna().sum()

print("\n── Summary ───────────────────────────────────────────")
print(f"  Total rows:     {len(fc_gdf):,}")
print(f"  Matched rows:   {len(fc_gdf) - missing:,}")
print(f"  Unmatched rows: {missing:,}")

if fc_gdf["TransLine_Distance_Meters"].notna().any():
    print(f"  Min distance:   {fc_gdf['TransLine_Distance_Meters'].min():,.2f}")
    print(f"  Max distance:   {fc_gdf['TransLine_Distance_Meters'].max():,.2f}")

# ── 6. SAVE OUTPUT ────────────────────────────────────────────────────────────
print("\nSaving output...")

fc_gdf.to_file(output_path, layer="updated_fc", driver="GPKG")

print(f"Done → {output_path}")

Loading data...
  Feature class rows: 122
  CSV rows:           11

Cleaning keys...
  Unique CSV keys: 11

Building lookup table...
Applying distance values...

── Summary ───────────────────────────────────────────
  Total rows:     122
  Matched rows:   122
  Unmatched rows: 0
  Min distance:   89.86
  Max distance:   88,629.62

Saving output...
2026-04-15 16:28:05  Created 122 records
Done → C:\Users\KyleSteen.AzureAD\Documents\Transmission_Lines_Workspace\Alaska\Alaska_output.gpkg
